# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display  # keep for nice formatting
from openai import OpenAI
import httpx

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# check API key
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj-"):
    print("Wrong API key format!")
elif api_key.strip() != api_key:
    print("API key has extra spaces!")
else:
    print("API key found and looks good!")

# set up OpenAI client
openai = OpenAI()

API key found and looks good!


In [4]:
# system prompt
system_prompt = """You are a helpful assistant.
When given a question, provide a clear, 
simple and easy to understand explanation.
Use examples where helpful.
Format your response in Markdown."""

In [5]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [7]:
# Get gpt-4o-mini to answer, with streaming
print("GPT-4o-mini Answer 🤖")
print("-" * 50)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# Call OpenAI with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    temperature=0.7,
    stream=True
)

# Print streamed response
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="", flush=True)

GPT-4o-mini Answer 🤖
--------------------------------------------------
Certainly! Let's break down the code you provided step by step.

### Code Explanation

```python
yield from {book.get("author") for book in books if book.get("author")}
```

1. **Set Comprehension**:
   - The part `{book.get("author") for book in books if book.get("author")}` is a **set comprehension**.
   - It creates a set of authors from a list of `books`.

2. **Iterating through `books`**:
   - `books` is expected to be an iterable (like a list) containing dictionaries, where each dictionary represents a book.
   - Each dictionary may contain keys like `"author"`.

3. **Getting the Author**:
   - For each `book` in `books`, we call `book.get("author")`.
   - The `get` method retrieves the value associated with the key `"author"`. If the key doesn't exist, it returns `None`.

4. **Filtering Authors**:
   - The `if book.get("author")` part filters out any books that do not have an author listed (i.e., it only inc

In [8]:
# Get Llama 3.2 to answer
print("Llama 3.2 Answer 🦙")
print("-" * 50)

# set up Ollama client
llama_client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',
    http_client=httpx.Client(
        trust_env=False  # 👈 ignore system proxy settings
    )
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

# Call Llama
response = llama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    temperature=0.7
)

display(Markdown(response.choices[0].message.content))


Llama 3.2 Answer 🦙
--------------------------------------------------


**Code Explanation**
=====================

The given code uses a concept called **generator expression** in Python.

```python
yield from {book.get("author") for book in books if book.get("author")}
```

Let's break it down:

* `for book in books`: This part iterates over a list or iterable of objects (`books`) and assigns each object to the variable `book`.
* `if book.get("author")`: This condition filters out any `book` that doesn't have an `"author"` key. If a `book` has an `"author"`, it will be included in the iteration.
* `{... for book in ... if ...}`: This is a generator expression, which is similar to a list comprehension but returns an iterator instead of a list.
* `yield from`: This keyword is used to yield multiple values from a sub-iterator. It's like saying "yield all the values from this sub-iterator".
* `{book.get("author") for book in ... if ...}`: This part gets the `"author"` value from each `book` that passes the filter condition (`if`). The resulting values are yielded by the generator expression.

**What does it do?**
--------------------

In summary, this code generates an iterator that yields all the authors of books in the `books` list. If a book doesn't have an `"author"`, its value will be `None`.

Here's an example to illustrate this:

```python
books = [
    {"title": "Book 1", "author": "John"},
    {"title": "Book 2", "author": "Jane"},
    {"title": "Book 3"}
]

authors = yield from {book.get("author") for book in books if book.get("author")}
print(authors)  # Output: ['John', 'Jane']
```

In this example, the code generates an iterator that yields the authors of books `"Book 1"` and `"Book 2"`. The value for `"Book 3"` is `None` because it doesn't have an `"author"` key.